# IDM Bulk Data Pull — concurrent extraction + analysis

Feed an input CSV of ids into a `ThreadPoolExecutor` fan-out of IDM GET calls (`requests` session + `urllib.parse` URLs + `urllib3` retries + cross-thread rate limiter), get rich execution-summary tables, success/failed JSON+CSV outputs — and the result as a pandas DataFrame for identity-resolution analysis.

> Style follows a production ForgeRock extraction workflow (env-driven config, rich summary tables), rewritten as an importable library. All hosts/credentials/ids here are fake.

## 1. Imports — `requests`, `urllib`, `concurrent.futures` (inside the lib), `tqdm`, `rich`, `pandas`.

In [ ]:
import os

import pandas as pd
from rich.console import Console

from idm_pull.extract import load_config, run_extraction
from idm_pull import analysis

console = Console()
pd.set_option('display.max_columns', 50)

## 2. Configuration — environment variables
No credentials in the notebook. Set these in your shell (or a `.env` you never commit):
`IDM_BASE_URL`, `IDM_USERNAME`, `IDM_PASSWORD`, `IDM_INPUT_FILE`, `IDM_INPUT_COLUMN`, `IDM_FIELDS` (comma-separated), `IDM_MAX_THREADS`, `IDM_VERIFY_SSL`, `IDM_RATE_LIMIT`.

In [ ]:
# point these at your environment, then run the rest
os.environ['IDM_BASE_URL']   = 'https://idm.example.com:8443/openidm/managed/user'
os.environ['IDM_USERNAME']   = 'svc-data-extract'
os.environ['IDM_PASSWORD']   = 'REPLACE-ME'          # real runs: export this, don't paste it
os.environ['IDM_INPUT_FILE'] = 'notebooks/input_ids.example.csv'
os.environ['IDM_FIELDS']     = 'userName,mail,givenName,sn,employeeNumber,accountStatus,city,country'
os.environ['IDM_MAX_THREADS'] = '10'

cfg = load_config()
print('pulling', len(cfg['fields']), 'fields with', cfg['max_threads'], 'threads')
print('fields:', cfg['fields'])

## 3. Run the extraction
`run_extraction()` validates config → loads input ids → fans out `ThreadPoolExecutor` GETs with a `tqdm` bar → writes `success/failed/execution_metadata/failure_summary` as JSON + CSV into an output folder named after the input file → prints rich Execution Summary and Failure Breakdown tables → returns the success DataFrame.

In [ ]:
df = run_extraction(cfg)
print(df.shape)
df.head()

## 4. Analysis — duplicate / fake-profile identity resolution
Same mail or employeeNumber on multiple records is the classic duplicate/fake-profile signal.

In [ ]:
# how complete is each pulled field?
analysis.completeness(df)

In [ ]:
summary, dup_rows = analysis.duplicates(df, 'mail')
console.print(f'[bold]{len(summary)} duplicated mail values[/bold]')
summary.head(10)

In [ ]:
summary_emp, _ = analysis.duplicates(df, 'employeeNumber')
console.print(f'[bold]{len(summary_emp)} duplicated employeeNumber values[/bold]')
summary_emp.head(10)

In [ ]:
# fetch failures live in failed.csv next to success.csv — quick peek
analysis.errors(df)

## 5. Next steps
- Join the dup leads against HR feeds, or feed them into the [forgerock-data-toolkit](https://github.com/vamsh1x/forgerock-data-toolkit) remediator (dry-run first).
- Correlate OTP-burst phone numbers from `twilio_otp_analysis.ipynb` with `telephoneNumber` here.
- Pull AM/IDM error logs for the failing windows with `splunk_data_analysis.ipynb`.

**Safety:** respect the rate limiter, use least-privilege service accounts, and never commit real credentials — `.gitignore` already excludes `*_config.json` and output folders.